# 04 — Baselines and rolling-origin evaluation

This notebook evaluates Dataset A: one official national ATP vacancy series and four non-total major-region series. Every forecast is generated at an explicit origin using only labels available by that origin. Required models are seasonal naive, last value, and Ridge regression over the configured lag window.

In [ ]:
import hashlib, json, math, os
from pathlib import Path
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
import yaml
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def _find_repo():
    env = os.environ.get("JOBAI_REPO")
    if env:
        return Path(env).resolve()
    p = Path.cwd().resolve()
    for candidate in (p, *p.parents):
        if (candidate / "configs" / "eval.yaml").is_file():
            return candidate
    return p

REPO = _find_repo()
PRO = REPO / "data" / "processed"
REPORTS = REPO / "reports"
FIGURES = REPORTS / "figures"
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
CFG = yaml.safe_load((REPO / "configs" / "eval.yaml").read_text())
HORIZONS = [int(h) for h in CFG["horizons"]]
WINDOW = int(CFG["feature_window_quarters"])
SEED = int(CFG["seed"])
np.random.seed(SEED)
print("repo:", REPO)
print("horizons:", HORIZONS, "window:", WINDOW)

## Load the five selected benchmark series

In [ ]:
selection_path = PRO / "selected_series.csv"
assert selection_path.is_file(), "Run notebook 03 first"
selection = pd.read_csv(selection_path)
benchmark_rows = selection[(selection["role"] == "benchmark_target") & selection["selected"].astype(bool)].copy()
assert len(benchmark_rows) == 5, f"Expected five Dataset A series, found {len(benchmark_rows)}"

def quarter_ordinal(q):
    return int(q[:4]) * 4 + int(q[-1]) - 1

series_data = {}
for row in benchmark_rows.itertuples(index=False):
    dims = json.loads(row.dimensions_json)
    frame = pd.read_csv(PRO / f"{row.table_id}__normalized.csv", low_memory=False)
    for column, value in dims.items():
        frame = frame[frame[column].astype(str) == str(value)]
    frame = frame[["timeperiod_q", "value"]].copy()
    frame["value"] = pd.to_numeric(frame["value"], errors="coerce")
    frame["quarter_index"] = frame["timeperiod_q"].map(quarter_ordinal)
    frame = frame.sort_values("quarter_index").reset_index(drop=True)
    assert frame["timeperiod_q"].is_unique
    assert frame["value"].notna().all(), f"Dataset A contains nulls: {row.series_id}"
    assert np.diff(frame["quarter_index"]).tolist() == [1] * (len(frame) - 1), f"Dataset A has a time gap: {row.series_id}"
    series_data[row.series_id] = {"table_id": row.table_id, "frame": frame}
print("benchmark series loaded:", len(series_data))

## Expanding-window forecasts

For a forecast made at origin `t`, Ridge training examples are restricted to examples whose target quarter is no later than `t`. Seasonal naive predicts `y[t+h-4]`, which is available at the origin for the configured horizons.

In [ ]:
splits = CFG["splits"]

def split_for_origin(q):
    if splits["val_start"] <= q <= splits["val_end"]:
        return "validation"
    if splits["test_start"] <= q <= splits["test_end"]:
        return "test"
    return None

prediction_rows = []
for series_id, info in series_data.items():
    frame = info["frame"]
    quarters = frame["timeperiod_q"].tolist()
    values = frame["value"].to_numpy(dtype=float)
    for origin_idx in range(WINDOW - 1, len(values)):
        origin = quarters[origin_idx]
        split = split_for_origin(origin)
        if split is None:
            continue
        history = values[:origin_idx + 1]
        seasonal_diffs = np.abs(history[4:] - history[:-4])
        mase_scale = float(np.mean(seasonal_diffs))
        assert mase_scale > 0
        for horizon in HORIZONS:
            target_idx = origin_idx + horizon
            if target_idx >= len(values):
                continue
            y_true = float(values[target_idx])
            forecasts = {
                "last_value": float(values[origin_idx]),
                "seasonal_naive": float(values[origin_idx + horizon - 4]),
            }
            X_train, y_train = [], []
            latest_train_target = None
            for train_origin in range(WINDOW - 1, origin_idx - horizon + 1):
                train_target = train_origin + horizon
                assert train_target <= origin_idx
                X_train.append(values[train_origin - WINDOW + 1:train_origin + 1])
                y_train.append(values[train_target])
                latest_train_target = train_target
            assert len(X_train) >= 10, f"Too few Ridge examples for {series_id}, {origin}, h={horizon}"
            ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
            ridge.fit(np.asarray(X_train), np.asarray(y_train))
            forecasts["ridge"] = float(ridge.predict(values[origin_idx - WINDOW + 1:origin_idx + 1].reshape(1, -1))[0])
            for model, y_pred in forecasts.items():
                abs_error = abs(y_true - y_pred)
                denom = (abs(y_true) + abs(y_pred)) / 2.0
                prediction_rows.append({
                    "table_id": info["table_id"], "series_id": series_id, "split": split,
                    "model": model, "horizon_q": horizon, "origin_quarter": origin,
                    "target_quarter": quarters[target_idx], "y_true": y_true, "y_pred": y_pred,
                    "error": y_true - y_pred, "abs_error": abs_error, "squared_error": (y_true - y_pred) ** 2,
                    "mase_scale": mase_scale, "scaled_abs_error": abs_error / mase_scale,
                    "smape_component_pct": 0.0 if denom == 0 else 100.0 * abs_error / denom,
                    "max_train_target_quarter": quarters[latest_train_target] if model == "ridge" else origin,
                })
predictions = pd.DataFrame(prediction_rows)
assert not predictions.empty
assert not predictions.duplicated(["series_id", "split", "model", "horizon_q", "origin_quarter"]).any()
assert (predictions["target_quarter"].map(quarter_ordinal) > predictions["origin_quarter"].map(quarter_ordinal)).all()
assert (predictions["max_train_target_quarter"].map(quarter_ordinal) <= predictions["origin_quarter"].map(quarter_ordinal)).all()
print("prediction rows:", len(predictions))

In [ ]:
def metric_summary(group):
    return pd.Series({
        "n_forecasts": len(group),
        "MAE": group["abs_error"].mean(),
        "RMSE": math.sqrt(group["squared_error"].mean()),
        "MASE": group["scaled_abs_error"].mean(),
        "sMAPE_pct": group["smape_component_pct"].mean(),
    })

by_series = (predictions.groupby(["split", "series_id", "model", "horizon_q"], sort=True)
             .apply(metric_summary, include_groups=False).reset_index())
summary = (predictions.groupby(["split", "model", "horizon_q"], sort=True)
           .apply(metric_summary, include_groups=False).reset_index())
counts = predictions.groupby(["split", "model", "horizon_q"]).size().unstack(["model", "horizon_q"] )
assert counts.notna().all().all(), "A baseline is missing from an evaluated split/horizon"
display(summary.round(3))

In [ ]:
prediction_path = REPORTS / "baseline_predictions.csv"
series_metrics_path = REPORTS / "baseline_metrics_by_series.csv"
summary_path = REPORTS / "baselines.csv"
predictions.to_csv(prediction_path, index=False)
by_series.to_csv(series_metrics_path, index=False)
summary.to_csv(summary_path, index=False)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

manifest = {
    "dataset": "Dataset A: 11l1 national + four non-total 11n1 regions",
    "series_count": len(series_data), "horizons_q": HORIZONS, "feature_window_quarters": WINDOW,
    "models": ["seasonal_naive", "last_value", "ridge"], "splits": CFG["splits"],
    "information_cutoff_assertion": "passed",
    "outputs": {str(path.relative_to(REPO)): {"rows": len(frame), "sha256": sha256(path)} for path, frame in [
        (prediction_path, predictions), (series_metrics_path, by_series), (summary_path, summary)
    ]},
}
manifest_path = REPORTS / "baseline_run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print("wrote:", prediction_path)
print("wrote:", series_metrics_path)
print("wrote:", summary_path)
print("wrote:", manifest_path)

In [ ]:
test_mae = summary[summary["split"] == "test"].pivot(index="horizon_q", columns="model", values="MAE")
ax = test_mae.plot(kind="bar", figsize=(10, 5))
ax.set(title="Dataset A rolling-origin test MAE", xlabel="Forecast horizon (quarters)", ylabel="MAE")
ax.tick_params(axis="x", rotation=0)
ax.figure.tight_layout()
ax.figure.savefig(FIGURES / "04_test_mae_by_horizon.png", dpi=150, bbox_inches="tight")
plt.show()